In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# 定义CNN
class CNN(nn.Module):  # 继承nn.Module
    def __init__(self):
        super().__init__()

        # Block1
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2)

        # Block2
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2)

        # 展平
        self.flatten = nn.Flatten()

        # 全连接层
        self.fc1 = nn.Linear(16 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        
        # Block1
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        # Block2
        x = F.relu(self.conv2(x))
        x = self.pool2(x)

        # Flatten + FC
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [ ]:
# 定义训练循环
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

def train(model, train_loader, test_loader, epochs=5, lr=0.01):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = {'loss' : [], 'acc' : []}

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            # 核心五步
            optimizer.zero_grad()
            preds = model(x)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        acc = evaluate(model, test_loader)

        history['loss'].append(avg_loss)
        history['acc'].append(acc)

        print(f'epoch:{epoch+1}/{epochs} | loss:{avg_loss} | accuray=cy:{acc}')

    return history


model = CNN()
train(model, train_loader, test_loader, epochs=5, lr=0.01)